# pymdp 基礎チュートリアル
## 能動的推論（Active Inference）の理論とコード

このノートブックでは、本研究プロジェクト（AIF Calibration Robustness）の理論的基盤を  
pymdp の最小サンプル「T迷路」を動かしながら理解します。

### 学習の流れ
1. 変分自由エネルギー（VFE）とは何か
2. 生成モデルの4行列（A, B, C, D）の設計
3. T迷路エージェントの実装と実行
4. 精度重み（Π）の効果を確認
5. 本研究（視触覚クロスモーダル）への橋渡し

---
## 1. 変分自由エネルギー（VFE）とは

能動的推論の核心は「エージェントは変分自由エネルギー F を最小化するように動く」という原理です。

$$F = \underbrace{D_{KL}[q(s) \| p(s)]}_{\text{複雑さ}} - \underbrace{\mathbb{E}_{q(s)}[\log p(o|s)]}_{\text{精度（観測の説明能力）}}$$

- **$q(s)$**：エージェントが持つ世界の信念（近似事後分布）
- **$p(s)$**：世界の事前モデル
- **$p(o|s)$**：状態 s から観測 o が生まれるモデル（観測モデル = A 行列）

F を最小化するということは：
- **知覚**：観測データに合うように信念 q(s) を更新する
- **行動**：予測した感覚に世界を一致させる（行動による F 最小化）

### 強化学習との違い
| | 強化学習（RL） | 能動的推論（AIF） |
|---|---|---|
| 目標 | 報酬の最大化 | 自由エネルギーの最小化（= 驚きの最小化） |
| 探索 | ε-greedy 等で手動設計 | EFE の「エピステミック項」として自動生成 |
| 不確実性 | 扱いが難しい | 精度重み Π として明示的に表現 |

---
## 2. 生成モデルの4行列

pymdp では世界を POMDP（部分観測マルコフ決定過程）としてモデル化します。  
エージェントは以下の4行列で「世界の内部モデル」を持ちます：

| 行列 | 形状 | 意味 | 本研究での対応 |
|---|---|---|---|
| **A** | `[num_obs, num_states]` | 観測モデル $p(o\|s)$ | 視覚・触覚がどう生成されるか |
| **B** | `[num_states, num_states, num_actions]` | 遷移モデル $p(s'\|s,a)$ | グリッパ動作で物体状態がどう変わるか |
| **C** | `[num_obs]` | 好ましい観測（log事前選好） | 「把持成功」状態を好む |
| **D** | `[num_states]` | 状態の事前分布 | 物体位置が不明な初期信念 |

---
## 3. T迷路問題の設定

```
    [報酬L]   [報酬R]
       |         |
      [左]     [右]
        \       /
         [中央] ← エージェントはここからスタート
           |
          [手がかり] ← ここで「報酬がどちらにあるか」のヒントを観測できる
```

- エージェントはまず手がかり位置に移動して報酬の場所を確認（エピステミック行動）
- 次に報酬のある方向へ移動（実利的行動）
- これが「好奇心と目標達成が EFE から自動的に生まれる」ことの最小デモ

### 状態と観測の設計
**隠れ状態（2つの因子）：**
- 因子0：エージェントの位置（中央=0, 左=1, 右=2, 手がかり=3）
- 因子1：報酬の場所（左=0, 右=1）

**観測（2つのモダリティ）：**
- モダリティ0：位置観測（中央=0, 左=1, 右=2, 手がかり=3）
- モダリティ1：報酬観測（中立=0, 報酬=1, 罰=2）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 4)
# Graphs use English only — DejaVu Sans is sufficient
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

# pymdp v1.0 は JAX バックエンドに刷新されたため、
# 従来の numpy ベース API は pymdp.legacy に格納されています
from pymdp.legacy import utils
from pymdp.legacy.agent import Agent

print(f'pymdp import OK (using pymdp.legacy API for v1.0 compatibility)')
print(f'numpy version: {np.__version__}')

### A行列の設計（観測モデル）
「状態 s のとき、観測 o がどのくらいの確率で生まれるか」を定義します。

In [ ]:
# 状態空間の定義
num_states = [4, 2]        # 因子0: 位置(4種), 因子1: 報酬場所(2種)
num_obs    = [4, 3]        # モダリティ0: 位置観測(4種), モダリティ1: 報酬観測(3種)
num_actions = [4]          # 行動: 中央/左/右/手がかりへの移動
num_factors = len(num_states)
num_modalities = len(num_obs)

# A行列を初期化（各モダリティについて）
A = utils.obj_array_zeros([[o] + num_states for o in num_obs])

# --- モダリティ0: 位置観測（位置は確実に観測できる）---
# 位置因子の値がそのまま位置観測になる
for loc in range(num_states[0]):
    for reward_cond in range(num_states[1]):
        A[0][loc, loc, reward_cond] = 1.0  # 確率1で位置が分かる

# --- モダリティ1: 報酬観測（場所によって異なる）---
# 位置0（中央）: 報酬情報なし → 「中立」を観測
A[1][0, 0, :] = 1.0   # 中央では常に「中立」

# 位置1（左）: 報酬場所が左(0)なら「報酬」、右(1)なら「罰」
A[1][1, 1, 0] = 1.0   # 左にいて報酬が左 → 報酬
A[1][2, 1, 1] = 1.0   # 左にいて報酬が右 → 罰

# 位置2（右）: 報酬場所が右(1)なら「報酬」、左(0)なら「罰」
A[1][2, 2, 0] = 1.0   # 右にいて報酬が左 → 罰
A[1][1, 2, 1] = 1.0   # 右にいて報酬が右 → 報酬

# 位置3（手がかり）: 報酬場所を直接教えてくれる
A[1][1, 3, 0] = 1.0   # 手がかり位置で報酬が左 → 「報酬(左を示唆)」
A[1][2, 3, 1] = 1.0   # 手がかり位置で報酬が右 → 「罰(右を示唆)」

print('A[0] shape:', A[0].shape, '  (位置観測モデル)')
print('A[1] shape:', A[1].shape, '  (報酬観測モデル)')
print('\n位置観測 A[0][:, :, 0] (報酬が左の場合):')
print(A[0][:, :, 0])

In [ ]:
# B行列の設計（状態遷移モデル）
# 「行動 a をとったとき、状態 s が s' に変わる確率」

B = utils.obj_array(num_factors)

# 因子0（位置）: 行動に応じて位置が確実に変わる
B[0] = np.zeros((num_states[0], num_states[0], num_actions[0]))
# 行動: 0=中央へ, 1=左へ, 2=右へ, 3=手がかりへ
for action in range(num_actions[0]):
    B[0][action, :, action] = 1.0   # 行動iで位置iへ確実に移動

# 因子1（報酬場所）: 行動によらず変わらない（環境の隠れた特性）
B[1] = np.zeros((num_states[1], num_states[1], num_actions[0]))
for action in range(num_actions[0]):
    B[1][:, :, action] = np.eye(num_states[1])  # 報酬場所は変わらない

print('B[0] shape:', B[0].shape, '  (位置遷移モデル)')
print('B[1] shape:', B[1].shape, '  (報酬場所遷移モデル = 不変)')

In [ ]:
# C行列の設計（好ましい観測 = 目標）
# 「どの観測が好ましいか」を log 確率で定義

C = utils.obj_array_zeros(num_obs)

# モダリティ0（位置観測）: 場所に対する選好なし
C[0] = np.zeros(num_obs[0])

# モダリティ1（報酬観測）: 報酬(1)を好み、罰(2)を嫌う
# 値が大きいほど「好ましい」（log確率に対応）
C[1][0] =  0.0   # 中立: 無関心
C[1][1] =  2.0   # 報酬: 強く好む
C[1][2] = -2.0   # 罰:   強く嫌う

print('C[1] (報酬観測への選好):', C[1])
print('  中立=0.0, 報酬=+2.0（好む）, 罰=-2.0（嫌う）')

In [ ]:
# D行列の設計（状態の事前分布 = 初期信念）
# 「タスク開始時に自分がどの状態にいると思っているか」

D = utils.obj_array_uniform(num_states)

# 位置は中央(0)にいることが確実
D[0] = np.array([1.0, 0.0, 0.0, 0.0])   # 中央にいる

# 報酬場所はどちらか分からない（一様分布）
D[1] = np.array([0.5, 0.5])              # 左か右か不明

print('D[0] (初期位置信念):', D[0], '← 中央にいると確信')
print('D[1] (報酬場所の初期信念):', D[1], '← どちらか不明')

---
## 4. エージェントの実行

4行列が揃ったので、`Agent` を作成してシミュレーションを走らせます。  
pymdp の `Agent` が内部でやること：
1. `infer_states(obs)` → VFE を最小化して $q(s)$ を更新
2. `infer_policies()` → EFE $G(\pi)$ を計算して方策の分布 $q(\pi)$ を求める
3. `sample_action()` → $q(\pi)$ からサンプリングして行動を決定

In [ ]:
# エージェントの作成
# policy_len=2: 2ステップ先まで計画（手がかり → 報酬）
agent = Agent(
    A=A, B=B, C=C, D=D,
    policy_len=2,         # 計画の深さ
    inference_horizon=2,  # 推論の時間的深さ
)
print('エージェント作成完了')
print(f'  状態空間: {num_states}')
print(f'  観測空間: {num_obs}')
print(f'  行動空間: {num_actions}')

In [ ]:
# 環境の定義（真の状態）
# 「実際には報酬は左(0)にある」とします
true_reward_location = 0  # 左に報酬

def get_observation(location, reward_location):
    """真の状態から観測を生成する（環境側の処理）"""
    obs_location = location
    
    if location == 0:    # 中央: 報酬情報なし
        obs_reward = 0
    elif location == 3:  # 手がかり: 報酬場所を教える
        obs_reward = 1 if reward_location == 0 else 2
    elif location == 1:  # 左
        obs_reward = 1 if reward_location == 0 else 2
    elif location == 2:  # 右
        obs_reward = 1 if reward_location == 1 else 2
    
    return [obs_location, obs_reward]

location_names = ['中央', '左', '右', '手がかり']
reward_obs_names = ['中立', '報酬', '罰']
action_names = ['→中央', '→左', '→右', '→手がかり']

print(f'真の報酬場所: {location_names[true_reward_location]}')

In [ ]:
# シミュレーション実行
T = 3  # タイムステップ数
current_location = 0  # 中央からスタート

print('=' * 50)
print('T迷路シミュレーション開始')
print(f'真の報酬場所: {location_names[true_reward_location]}')
print('=' * 50)

history = []

for t in range(T):
    # 1. 現在位置から観測を取得
    obs = get_observation(current_location, true_reward_location)
    
    # 2. VFE最小化: 観測から状態を推論
    qs = agent.infer_states(obs)
    
    # 3. EFE計算: 方策を推論
    q_pi, G = agent.infer_policies()
    
    # 4. 行動サンプリング
    action = agent.sample_action()
    action_taken = int(action[0])
    
    # 5. 環境の更新
    current_location = action_taken
    
    # ログ
    history.append({
        't': t,
        'obs': obs,
        'belief_location': qs[0],
        'belief_reward': qs[1],
        'action': action_taken,
        'G': G
    })
    
    print(f'\nステップ {t+1}:')
    print(f'  観測: 位置={location_names[obs[0]]}, 報酬観測={reward_obs_names[obs[1]]}')
    print(f'  信念（報酬場所）: 左={qs[1][0]:.3f}, 右={qs[1][1]:.3f}')
    print(f'  選択行動: {action_names[action_taken]}')

print('\n' + '=' * 50)
print('シミュレーション終了')

In [ ]:
# Visualize simulation results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

steps = [h['t'] for h in history]

location_names_en = ['Center', 'Left', 'Right', 'Cue']

# Belief over reward location
belief_left  = [h['belief_reward'][0] for h in history]
belief_right = [h['belief_reward'][1] for h in history]
axes[0].plot(steps, belief_left,  'b-o', label='Reward at Left', linewidth=2)
axes[0].plot(steps, belief_right, 'r-o', label='Reward at Right', linewidth=2)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Time step')
axes[0].set_ylabel('Belief probability')
axes[0].set_title('Belief over reward location q(s1)')
axes[0].legend()
axes[0].set_ylim([0, 1])

# Belief over agent location
for loc in range(4):
    vals = [h['belief_location'][loc] for h in history]
    axes[1].plot(steps, vals, '-o', label=location_names_en[loc], linewidth=2)
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Belief probability')
axes[1].set_title('Belief over location q(s0)')
axes[1].legend(fontsize=8)
axes[1].set_ylim([0, 1])

# Action history
action_names_en = ['->Center', '->Left', '->Right', '->Cue']
actions = [h['action'] for h in history]
axes[2].bar(steps, actions, color='steelblue')
axes[2].set_xlabel('Time step')
axes[2].set_ylabel('Action index')
axes[2].set_yticks(range(4))
axes[2].set_yticklabels(action_names_en, fontsize=8)
axes[2].set_title('Selected actions')

plt.tight_layout()
plt.savefig('t_maze_result.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: t_maze_result.png')

---
## 5. 精度重み（Π）の効果を確認

本研究の核心は「視覚の精度 $\Pi_{visual}$ が低いとき（オクルージョン）、  
触覚の精度 $\Pi_{tactile}$ を上げて行動を切り替える」メカニズムです。

T迷路で「手がかりモダリティの精度を下げる」ことで、  
情報量の少ない観測に対してエージェントが信念を更新しにくくなることを確認します。

### A行列に精度重みを適用する

pymdp での精度重みの扱い方：  
A 行列に noisy（曖昧）なバージョンを渡すことで「センサーが信頼できない」状態をモデル化します。

In [ ]:
def add_noise_to_A(A_clean, modality_idx, noise_level):
    """
    精度重みの低下を「ノイズ追加」でモデル化
    noise_level = 0.0: 完全信頼（精度=最大）
    noise_level = 1.0: 完全ランダム（精度=最小）
    """
    import copy
    A_noisy = copy.deepcopy(A_clean)
    num_obs_m = A_clean[modality_idx].shape[0]
    uniform = np.ones_like(A_clean[modality_idx]) / num_obs_m
    A_noisy[modality_idx] = (1 - noise_level) * A_clean[modality_idx] + noise_level * uniform
    return A_noisy

# 条件1: 通常（精度=高、ノイズ=なし）
# 条件2: 手がかり観測が不明瞭（精度=低、ノイズ=大）
noise_levels = [0.0, 0.5, 0.9]
results = {}

for noise in noise_levels:
    A_noisy = add_noise_to_A(A, 1, noise)  # モダリティ1（報酬観測）にノイズ
    agent_noisy = Agent(A=A_noisy, B=B, C=C, D=D, policy_len=2, inference_horizon=2)
    
    loc = 0
    beliefs_over_time = []
    for t in range(3):
        obs = get_observation(loc, true_reward_location)
        qs = agent_noisy.infer_states(obs)
        q_pi, G = agent_noisy.infer_policies()
        action = agent_noisy.sample_action()
        loc = int(action[0])
        beliefs_over_time.append(qs[1].copy())
    
    results[noise] = beliefs_over_time
    print(f'ノイズ={noise:.1f}: 最終ステップの報酬場所信念 左={beliefs_over_time[-1][0]:.3f}, 右={beliefs_over_time[-1][1]:.3f}')

In [ ]:
# Visualize precision weight effect
fig, axes = plt.subplots(1, len(noise_levels), figsize=(14, 4), sharey=True)

for i, noise in enumerate(noise_levels):
    b = results[noise]
    axes[i].bar(['Left (correct)', 'Right'], [b[-1][0], b[-1][1]], color=['steelblue', 'salmon'])
    axes[i].set_title(f'Observation noise = {noise}\n(Precision Pi ~ {1-noise:.1f})')
    axes[i].set_ylabel('Belief probability over reward location')
    axes[i].set_ylim([0, 1])
    axes[i].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Uncertain')

plt.suptitle('Effect of precision (Pi): higher noise -> weaker belief update', fontsize=12)
plt.tight_layout()
plt.savefig('precision_effect.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  Noise=0.0 (Pi=high): trust cue  -> confident "reward at Left"')
print('  Noise=0.5 (Pi=mid) : half trust -> weak belief update')
print('  Noise=0.9 (Pi=low) : ignore cue -> stays near prior (50/50)')

---
## 6. 本研究への橋渡し

T迷路で学んだことが、本研究（AIF Calibration Robustness）にどう対応するか整理します。

### T迷路 ↔ 視触覚クロスモーダルAIFの対応

| T迷路の概念 | 本研究での対応 |
|---|---|
| 手がかり位置への移動 | オクルージョン発生 → 触覚センサーへの依存 |
| 手がかりモダリティ（モダリティ1） | 視覚モダリティ |
| 報酬観測（左/右） | 把持対象の位置・素材推定 |
| ノイズレベルの増加 | オクルージョン度合い（c_visual の低下） |
| A行列のノイズ注入 | Π_visual の低下 |
| 触覚モダリティ（存在しなかった部分） | **←本研究が追加する新モダリティ** |

### 次のステップ（フェーズ1）

1. **2-DoFアームのシミュレーション**でAIFを実装
   - 関節角を離散化（20段階など）
   - 視覚：カメラ画像をエントロピーで信頼度スコア化
   - 触覚：サーボ電流値を暫定「触覚信号」として使用

2. **精度重み切り替えのデモ**
   - オクルージョンなし: Π_visual=高、Π_tactile=低
   - オクルージョンあり: Π_visual=低（ノイズ注入）、Π_tactile=高

### 本研究の目的関数（再確認）

$$F_{total} = \Pi_{visual} \cdot \varepsilon_{visual}^2 + \Pi_{tactile} \cdot \varepsilon_{tactile}^2 + \Pi_{proprio} \cdot \varepsilon_{proprio}^2$$

T迷路での「ノイズ注入」が $\Pi_{visual}$ の低下に対応していました。  
本研究では $\Pi_{tactile}$ を同時に「上げる」ことで、触覚主導の制御を実現します。

In [ ]:
# Conceptual demo: precision switching as a function of visual confidence
theta = 0.4  # Occlusion threshold

c_visual_range = np.linspace(0, 1, 100)

pi_visual  = np.where(c_visual_range >= theta,
                      1.0,
                      np.maximum(0.1, 1.0 - (theta - c_visual_range) / theta))

pi_tactile = np.where(c_visual_range >= theta,
                      1.0,
                      np.minimum(5.0, 1.0 + ((theta - c_visual_range) / theta) * 4.0))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(c_visual_range, pi_visual,  'b-',  linewidth=2.5, label='Pi_visual (visual precision)')
ax.plot(c_visual_range, pi_tactile, 'r-',  linewidth=2.5, label='Pi_tactile (tactile precision)')
ax.axvline(theta, color='gray', linestyle='--', linewidth=1.5, label=f'Threshold theta={theta}')
ax.fill_betweenx([0, 5.5], 0, theta, alpha=0.1, color='red', label='Occlusion region')
ax.set_xlabel('Visual confidence score c_visual')
ax.set_ylabel('Precision weight Pi')
ax.set_title('Precision switching triggered by occlusion\n(Johansson-Flanagan type)')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 5.5])
plt.tight_layout()
plt.savefig('precision_switching.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'c_visual=1.0 (fully visible)   : Pi_visual={pi_visual[-1]:.1f}, Pi_tactile={pi_tactile[-1]:.1f}')
print(f'c_visual=0.2 (strong occlusion): Pi_visual={pi_visual[20]:.2f}, Pi_tactile={pi_tactile[20]:.2f}')
print(f'c_visual=0.0 (fully occluded)  : Pi_visual={pi_visual[0]:.2f}, Pi_tactile={pi_tactile[0]:.2f}')

---
## まとめ

このノートブックで学んだこと：

1. **VFE** = 複雑さ（KL項）− 精度（尤度項） を最小化するのがAIFの核心
2. **4行列（A, B, C, D）** で生成モデルを定義する
3. **pymdp** の `infer_states → infer_policies → sample_action` の3ステップ
4. **精度重みΠ** = A行列のノイズレベルで表現できる（低精度 = 高ノイズ）
5. **本研究の核心** = c_visual 低下 → Π_visual↓ + Π_tactile↑ の自動切り替え

### 次のノートブック
`02_generative_model_design.ipynb` では、SO-101アーム + AnySkin を想定した  
クロスモーダル生成モデルの設計を行います。